# Typhoon Dataset Consolidation and Station Mapping

This notebook is designed to consolidate multiple typhoon information datasets into a single unified dataset. The original data is organized by year, with each file containing records for a specific time period. To facilitate easier analysis, the yearly datasets are merged into one comprehensive dataset that contains all available records.

In addition to combining the datasets, this notebook also appends **station location mapping information**. Weather station identifiers in the raw data are linked to their corresponding **province and region**, allowing the dataset to be analyzed geographically. This mapping enables the dataset to be aligned with other infrastructure and flood-related datasets used in the study.


**Disclaimer:**  
Generative AI tools were used to assist in converting typhoon report screenshots (sourced from **PAGASA**) into CSV format. AI tools were also used to generate an initial mapping between unique weather stations and their corresponding provinces and regions. The authors carefully reviewed, verified, and edited the generated content where necessary and take full responsibility for the accuracy and integrity of the final dataset.

## Import
Import **numpy**, **pandas**, and **glob**.

In [ ]:
# Import relevant python modules
import numpy as np
import pandas as pd
import glob

### Locating Yearly Typhoon Information Files

In this step, the notebook identifies all CSV files containing typhoon information that are stored in the **by-year** directory. Since the dataset is organized into separate files for each year, it is necessary to automatically gather all of these files before combining them into a single dataset. This more automated approach ensures that all yearly typhoon data files are included in the workflow without requiring each file to be manually specified.

In [ ]:
# Grab the paths of all CSV files in your folder
path = './data/typhoon-info/by-year' 
all_files = glob.glob(path + "/*.csv")

### Loading Multiple CSV Files into DataFrames

In this step, all the CSV files identified in the previous step are loaded into memory using a **list comprehension**. This approach allows multiple datasets to be read efficiently in a single line of code. By the end of this step, **li** contains a list of DataFrames, each representing one of the CSV files. These DataFrames can then be combined or further processed in the subsequent steps of the data cleaning workflow.

In [ ]:
# Use a list comprehension to read them all at once
li = [pd.read_csv(filename) for filename in all_files]

### Combining the Datasets into a Single DataFrame

After loading the individual CSV files into a list of DataFrames, the next step is to combine them into one unified dataset. This is done using the `pd.concat()` function, which merges multiple DataFrames together. At the end of this step, all records from the separate CSV files are consolidated into a single master dataset that can be cleaned and analyzed more efficiently.

In [ ]:
# Concatenate them into one master DataFrame
# Concatenate master
df = pd.concat(li, axis=0, ignore_index=True)

### Loading the Weather Station Location Mapping

In this step, the dataset containing the **weather station location mapping** is loaded into a pandas DataFrame. This mapping file links each unique weather station to its corresponding **province and region**, which allows the typhoon dataset to be analyzed geographically.

In [ ]:
# Clean it up before the merge
mapping_df = pd.read_csv('data/geospatial-data/station_location_mapping.csv')

### Removing Existing Geographic Columns Before Merging

Before merging the typhoon dataset with the station location mapping, any existing **Province** or **Region** columns are removed from the raw dataset. In some cases, these columns may already exist in the original data but may contain incomplete, inconsistent, or outdated values.

Keeping these columns during the merge process can lead to duplicated fields such as `Province_x`, `Province_y`, `Region_x`, and `Region_y`, which pandas automatically creates when overlapping column names are detected. These duplicated columns can make the dataset harder to interpret and clean.

To prevent this issue, the `drop()` method is used to remove the **Province** and **Region** columns if they are present. The parameter `errors='ignore'` ensures that the operation does not raise an error if these columns are not found in the dataset.

In [ ]:
# Remove geo columns from raw data if they exist to avoid the "x/y" issue
df = df.drop(columns=['Province', 'Region'], errors='ignore')

### Merging Typhoon Data with Station Location Mapping

In this step, the typhoon dataset is merged with the **station location mapping** dataset to append geographic information to each record. The mapping dataset contains the corresponding **province** and **region** for each weather station.

In [ ]:
# Merge
df = pd.merge(df, mapping_df, on='Location', how='left')

### Saving the Compiled Typhoon Dataset

After merging the yearly typhoon datasets and appending the station location mapping, the final cleaned dataset is exported to a CSV file. Saving the processed dataset allows it to be reused in later stages of the project without repeating the entire data cleaning and merging process.

In [ ]:
# Save your cleaned dataframe to a specific folder
# 'index=False' prevents pandas from adding an extra column of numbers at the start
df.to_csv('data/merged/typhoon-info/compiled_2019-2025.csv', index=False)